# 01. STITCHV2 Model, From Reads to Genotypes

This notebook explains the model before touching parameters. The short version is:

STITCHV2 models each individual chromosome as a mosaic of `K` founder haplotypes. At each variant, the hidden state says which founder copy or founder-copy combination generated the sample. The observed data are sequencing reads, read fragments, and optional hard genotype evidence from PLINK/microarray data. The HMM uses these observations to estimate genotype dosage and posterior genotype probabilities.

Original STITCH uses the same broad idea: a read-aware Li-Stephens-style HMM with founder haplotypes, EM updates, and low-coverage sequencing reads. STITCHV2 is not a line-by-line port; it is a Python/JAX implementation designed to keep the STITCH read-aware behavior while adding parquet IO, mixed ploidy, calibration, Dask orchestration, and Python API workflows.


In [1]:
from pathlib import Path
import os, sys, json, math, shutil, time

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
sys.path.insert(0, str(REPO / 'src'))
FIG_DIR = REPO / 'docs' / 'tutorial_deep' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_DIR = REPO / 'benchmark_runs' / 'tutorial_deep_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print('repo:', REPO)
print('synthetic data exists:', DATA_DIR.exists())


repo: /home/bonnie/Documents/codex/STITCHV2
synthetic data exists: True


## Hidden States

For a diploid sample, the optimized fast path uses ordered founder-pair states:

```text
z_i = (k_a, k_b), where k_a and k_b are founder indexes in 0..K-1
number of states = K^2
```

For haploid samples, the state is a single founder:

```text
z_i = k
number of states = K
```

For ploidy `P >= 3`, STITCHV2 uses unordered founder-count states:

```text
c_i = (c_0, c_1, ..., c_{K-1})
sum_k c_k = P
number of states = choose(K + P - 1, P)
```

That is much smaller than ordered states (`K^P`) while still representing all copy-count configurations.


In [2]:
from math import comb
rows = []
for K in [4, 8, 12, 16]:
    for P in [1, 2, 3, 4, 6]:
        fast_or_generic = K if P == 1 else (K*K if P == 2 else comb(K + P - 1, P))
        ordered = K ** P
        rows.append({'K_founders': K, 'ploidy': P, 'STITCHV2_states': fast_or_generic, 'ordered_states': ordered})
states = pd.DataFrame(rows)
display(states)

fig, ax = plt.subplots(figsize=(7, 4))
for K, sub in states[states.K_founders.isin([4, 8, 12])].groupby('K_founders'):
    ax.plot(sub['ploidy'], sub['STITCHV2_states'], marker='o', label=f'K={K}')
ax.set_yscale('log')
ax.set_xlabel('Ploidy')
ax.set_ylabel('Hidden states, log scale')
ax.set_title('Generic polyploid count-state growth')
ax.legend()
fig.tight_layout()
out = FIG_DIR / '01_state_count_growth.png'
fig.savefig(out, dpi=160)
print('wrote', out)


    K_founders  ploidy  STITCHV2_states  ordered_states
0            4       1                4               4
1            4       2               16              16
2            4       3               20              64
3            4       4               35             256
4            4       6               84            4096
5            8       1                8               8
6            8       2               64              64
7            8       3              120             512
8            8       4              330            4096
9            8       6             1716          262144
10          12       1               12              12
11          12       2              144             144
12          12       3              364            1728
13          12       4             1365           20736
14          12       6            12376         2985984
15          16       1               16              16
16          16       2              256         

![State-count growth](figures/01_state_count_growth.png)

## Emissions: How Reads Become HMM Evidence

At a variant, each founder `k` has an alternate-allele probability `f[k, i]`. For a ploidy-`P` state with founder counts `c[k]`, the expected alternate fraction is:

```text
alt_fraction_i(state) = sum_k c[k] * f[k, i] / P
expected_dosage_i(state) = P * alt_fraction_i(state)
```

A read that reports REF, ALT, or another base is converted to a probability using the sequencing error model and optional base/mapping quality weights. For a fragment/read covering several SNPs, STITCHV2 can couple those SNP observations rather than treating the center SNP alone. In STITCH-parity mode, the read contribution generalizes the diploid STITCH intuition:

```text
P(read | diploid state k1,k2) = 0.5 * P(read | k1) + 0.5 * P(read | k2)

P(read | polyploid count state c) = sum_k c[k] / P * P(read | k)
```

This is the key read-awareness point: reads and fragments contribute likelihood terms before the HMM posterior is decoded, not merely after a pileup is converted to hard genotypes.


## Transitions

Neighboring variants are connected by a recombination/switch probability. Higher `generation` values imply more opportunity for founder switches across a genomic distance, which makes the mosaic less sticky. Lower values imply longer founder tracts.

Original STITCH has one main `nGen` setting. STITCHV2 stores `generation` in the sample table, so different samples can have different transition scales.

For ploidy `P >= 3`, each homolog copy transitions independently. STITCHV2 builds count-state transition probabilities by aggregating those independent copy transitions into unordered count states.


## EM Updates

Each EM iteration has two conceptual steps:

1. **E-step:** run forward/backward and compute posterior probabilities over hidden founder states.
2. **M-step:** update mutable founder allele probabilities from expected founder-copy counts and read evidence.

If founders are immutable, STITCHV2 skips founder mutation and behaves closer to a reference-panel/parity mode. If founders are mutable, STITCHV2 can learn founder allele probabilities from the cohort.


## Outputs

The HMM produces dosage and optional posterior products:

- `dosage`: expected alternate allele count per sample and variant.
- `genotype_posteriors`: vector of length `P + 1`, one probability for alt count `0..P`.
- `genotype_calls`: hard call in `0..P`, or `-1` if a no-call policy rejects the posterior.
- `support_mask`: whether a sample/variant had direct read or microarray support.
- `founder_updates`, `recombination`, and optional haplotype probabilities for diagnostics.

The rest of the tutorial shows how to produce, inspect, calibrate, scale, and export these outputs.
